In [1]:
import pandas as pd
import numpy as np
import subprocess
import os
 
print(" Imports ready!")

 Imports ready!


In [4]:
# Checking all the files

files_needed = {
    'validation_predictions.csv':       'Our 12,000 predictions',
    'december_chart_inputs_filled.csv': 'Our 31 December predictions',
    'score.py':                         'The validation script'
}
 
print("CHECKING FILES EXIST:")
print("-"*60)
 
all_found = True
for fname, desc in files_needed.items():
    exists = os.path.isfile(fname)
    status = "Found" if exists else "MISSING"
    print(f"  {status}: {fname}")
    print(f"           ({desc})")
    if not exists:
        all_found = False
 
if all_found:
    print("\n All files found — ready to validate!")
else:
    print("\n Some files are missing — fix before continuing!")

CHECKING FILES EXIST:
------------------------------------------------------------
  Found: validation_predictions.csv
           (Our 12,000 predictions)
  Found: december_chart_inputs_filled.csv
           (Our 31 December predictions)
  Found: score.py
           (The validation script)

 All files found — ready to validate!


In [10]:
# Inspect Predictions before running score.py

preds = pd.read_csv('validation_predictions.csv')
 
print(f"\nShape:   {preds.shape}")
print(f"Columns: {preds.columns.tolist()}")
 
print(f"\nFirst 5 rows:")
print(preds.head())
 
print(f"\nLast 5 rows:")
print(preds.tail())
 
print(f"\nQuality checks:")
print(f"  Total rows:          {len(preds):,}")
print(f"  Missing load_ids:    {preds['load_id'].isnull().sum()}")
print(f"  Duplicate load_ids:  {preds['load_id'].duplicated().sum()}")
print(f"  Missing rates:       {preds['predicted_rate'].isnull().sum()}")
print(f"  Negative rates:      {(preds['predicted_rate'] <= 0).sum()}")
print(f"  Min rate:            ${preds['predicted_rate'].min():.2f}")
print(f"  Max rate:            ${preds['predicted_rate'].max():.2f}")
print(f"  Mean rate:           ${preds['predicted_rate'].mean():.2f}")


Shape:   (12000, 2)
Columns: ['load_id', 'predicted_rate']

First 5 rows:
     load_id  predicted_rate
0  TE-000001          842.52
1  TE-000002         4974.29
2  TE-000003         5941.54
3  TE-000004         4375.69
4  TE-000005         1795.77

Last 5 rows:
         load_id  predicted_rate
11995  TE-011996          789.30
11996  TE-011997         1443.93
11997  TE-011998          894.27
11998  TE-011999         3401.82
11999  TE-012000          508.44

Quality checks:
  Total rows:          12,000
  Missing load_ids:    0
  Duplicate load_ids:  0
  Missing rates:       0
  Negative rates:      0
  Min rate:            $172.48
  Max rate:            $8959.92
  Mean rate:           $2466.29


In [11]:
#Verify Load Ids format

EXPECTED_IDS  = {f"TE-{i:06d}" for i in range(1, 12001)}
submitted_ids = set(preds['load_id'].astype(str))
 
missing_ids = EXPECTED_IDS - submitted_ids
extra_ids   = submitted_ids - EXPECTED_IDS
 
print(f"\n Expected:     TE-000001 to TE-012000 ({len(EXPECTED_IDS):,} total)")
print(f"  Submitted:    {len(submitted_ids):,}")
print(f"  Missing IDs:  {len(missing_ids)}")
print(f"  Extra IDs:    {len(extra_ids)}")
print(f"  Sample IDs:   {sorted(list(submitted_ids))[:5]}")
 
if not missing_ids and not extra_ids:
    print(f"\n All 12,000 load_ids match exactly!")
else:
    print(f"\n ID mismatch — score.py will fail!")
    if missing_ids:
        print(f"  Missing: {list(missing_ids)[:5]}")
    if extra_ids:
        print(f"  Extra:   {list(extra_ids)[:5]}")


 Expected:     TE-000001 to TE-012000 (12,000 total)
  Submitted:    12,000
  Missing IDs:  0
  Extra IDs:    0
  Sample IDs:   ['TE-000001', 'TE-000002', 'TE-000003', 'TE-000004', 'TE-000005']

 All 12,000 load_ids match exactly!


In [14]:
# Run score.py for validation

print("""
Command:
  python3 score.py
    --predictions          validation_predictions.csv
    --december-predictions december_chart_inputs_filled.csv
    --output-dir           scorer_results
""")
 
result = subprocess.run([
    'python3', 'score.py',
    '--predictions',          'validation_predictions.csv',
    '--december-predictions', 'december_chart_inputs_filled.csv',
    '--output-dir',           'scorer_results'
], capture_output=True, text=True)
 
print(f"score.py output:")
print(f"  {result.stdout.strip()}")
 
if result.returncode == 0:
    print(f"\n score.py PASSED! Return code: {result.returncode}")
else:
    print(f"\n score.py FAILED! Return code: {result.returncode}")
    print(f"  Error: {result.stderr}")


Command:
  python3 score.py
    --predictions          validation_predictions.csv
    --december-predictions december_chart_inputs_filled.csv
    --output-dir           scorer_results

score.py output:
  Validated 12,000 final predictions.
Validated 31 fixed December predictions.
Created chart: scorer_results\candidate_december.png
Final validation metrics are calculated by Spotter after submission.

 score.py PASSED! Return code: 0


In [16]:
print("\nFINAL VALIDATION SUMMARY:")
print("="*60)
print(f"""
  File:                    validation_predictions.csv
  Total predictions:       {len(preds):,}
  Missing predictions:     {preds['predicted_rate'].isnull().sum()}
  Negative predictions:    {(preds['predicted_rate'] <= 0).sum()}
  Duplicate load_ids:      {preds['load_id'].duplicated().sum()}
  Missing load_ids:        {len(missing_ids)}
  Extra load_ids:          {len(extra_ids)}
  Min predicted rate:      ${preds['predicted_rate'].min():.2f}
  Max predicted rate:      ${preds['predicted_rate'].max():.2f}
  Mean predicted rate:     ${preds['predicted_rate'].mean():.2f}
  score.py validation:     {'PASSED' if result.returncode == 0 else '✗ FAILED'}
  Chart generated:         {'scorer_results/candidate_december.png' if result.returncode == 0 else '✗ Not generated'}
""")


FINAL VALIDATION SUMMARY:

  File:                    validation_predictions.csv
  Total predictions:       12,000
  Missing predictions:     0
  Negative predictions:    0
  Duplicate load_ids:      0
  Missing load_ids:        0
  Extra load_ids:          0
  Min predicted rate:      $172.48
  Max predicted rate:      $8959.92
  Mean predicted rate:     $2466.29
  score.py validation:     PASSED
  Chart generated:         scorer_results/candidate_december.png

